# DAG Mode — Notebook UI Design Spec

**Date:** 2026-05-31  ·  **Status:** Approved design, ready for implementation plan
**Area:** `crates/spur-notebook/jute-notebook` (frontend) + `crates/spur-notebook/src/dag` (engine)

## Summary

The notebook has two surfaces today — **Notebook mode** (linear cell list) and **Deck mode** (slides). This spec adds a third: **DAG mode**, a graph-first view of the notebook's *reactive data dependency graph*. Cells declare `produces` / `consumes` ports (`SpurCellMetadata.dag`); the new reactive engine (`crates/spur-notebook/src/dag`) derives edges, topological order, and stale-propagation from those ports. DAG mode visualizes that graph and lets the user edit + run nodes and watch the reactive cascade recolor the graph **live**.

DAG mode is an **in-place toggle inside Notebook mode** — not a separate route like Deck — so it shares the live `Notebook` store and event stream. Interactive behavioral prototype: `~/.spur/scratch/Untitled51.ipynb` (cell 5).

## Goals & Non-Goals

### Goals
- A graph-first **view** of the notebook DAG, as a peer surface to Notebook/Deck, reached by an in-place toggle.
- **Mode "B" interaction level**: select a node → see/edit its code in an inspector → run that node and trigger the reactive cascade, all without leaving the view.
- **Live** node/edge status: `fresh / stale / running / failed / upstream-failed / never-run`, driven by push events from the reactive engine (not polling).
- Auto-layout from graph structure (the graph is a pure projection of `produces/consumes`; no hand-placed coordinates to persist).
- Honest handling of non-DAG cells (markdown, portless scratch): hidden from the canvas, surfaced via a **hidden-cells chip**.

### Non-Goals (this iteration)
- **Structural authoring on the canvas** (drag-to-wire ports, create/delete cells from the graph). That is "mode C", a later effort.
- Persisted manual node positions.
- Multi-notebook / cross-file graphs.
- Replacing Notebook mode — DAG mode augments it.

## Approved Decisions

| # | Decision | Choice |
|---|----------|--------|
| 1 | View philosophy | **A** — graph-first canvas + inspector rail |
| 2 | Interaction level | **B** — run + light edit (single-node edit/run + cascade); no structural editing |
| 3 | Status freshness | **Push** — engine emits `dag_status_changed`; seed via `notebook_dag_status` on open |
| 4 | Layout | **Auto-layout** (dagre, top-down) — no persisted positions |
| 5 | Rendering lib | **`@xyflow/react` (React Flow)** + `@dagrejs/dagre` for layout math |
| 6 | Canvas membership | **DAG-only** nodes + a **hidden-cells chip** for portless/markdown cells |
| 7 | Entry | **In-place toggle** inside Notebook mode (`viewMode: 'cells' \| 'dag'`), not a new route |
| 8 | State vocabulary | `fresh`, `stale`, `running`, `failed`, `upstream-failed`, `never-run` |

## Architecture

```mermaid
flowchart LR
  subgraph FE["jute-notebook · React / TS"]
    Toggle["NotebookHeader<br/>viewMode toggle"] --> View["NotebookView"]
    View -->|"viewMode = cells"| Cells["NotebookCells<br/>(existing)"]
    View -->|"viewMode = dag"| DagView["DagView"]
    DagView --> Graph["useDagGraph()<br/>derive edges + dagre layout"]
    DagView --> NodeC["DagNode × N<br/>(custom RF node)"]
    DagView --> Insp["DagInspector<br/>CodeMirror + ports + run"]
    DagView --> Chip["HiddenCellsChip"]
    Store[("Notebook store<br/>zustand + immer")]
    Status["dagStatus: Map&lt;cellId, NodeStatus&gt;"]
    Store --- View
    Status --- DagView
  end
  subgraph BE["spur-notebook · Rust"]
    DagStatus["notebook_dag_status<br/>(snapshot: nodes/edges/port_manifest)"]
    RunCascade["run_cell_and_cascade*<br/>(NEW entrypoint)"]
    Engine["ReactiveEngine<br/>(debounce 150ms · max_in_flight 4)"]
    Events[("notebook event channel<br/>(existing FE subscription)")]
  end
  Insp -->|"run_cell / run_cell_and_cascade"| BE
  DagView -->|"seed on toggle-open"| DagStatus
  Engine -->|"dag_status_changed (push)"| Events
  Events --> Status
```

**Layer boundaries**
- `DagView` is the only component that knows about React Flow; everything else stays library-agnostic.
- `useDagGraph()` is a pure transform: cells + `dagMetadata` → `{nodes, edges}` + dagre positions. Mirrors Rust `NotebookDag::edges()` so FE and BE agree on graph shape. Independently testable.
- `dagStatus` is a separate slice keyed by cell id; the canvas reads it but the graph *shape* is independent of status, so status churn never relayouts.

## Data Model & Store Changes

DAG metadata already exists end-to-end as types but is **not plumbed into the frontend store**:

- Rust: `SpurCellMetadata.dag: Option<CellDagMetadata>` → persisted at `cell.metadata.spur.dag`.
- TS bindings (auto-generated): `CellDagMetadata { produces: PortSpec[]; consumes: string[]; source?: DagSource }`, `PortSpec { port; repr; display? }`, `DagSource { kind; port }`.
- Today `loadNotebookRootDraft` (in `stores/notebook.ts`) loads `juteDeckMetadata` but **skips** `dag`.

### Changes to `stores/notebook.ts`
1. Extend `NotebookCellState` with `dagMetadata?: CellDagMetadata`.
2. Populate it in `loadNotebookRootDraft` from `cell.metadata.spur.dag` (same spot deck metadata is read).
3. Add to `viewState`: `viewMode: 'cells' | 'dag'` (default `'cells'`).
4. Add a `dagStatus: Record<cellId, NodeStatus>` slice (separate from `serverState`; populated by seed + push).

```ts
type NodeStatus = {
  state: 'fresh' | 'stale' | 'running' | 'failed' | 'upstream-failed' | 'never-run';
  ranPortVersions: Record<string, number>; // consumed port versions at last successful run
  executionCount?: number;
};
```

**Stale derivation (frontend):** a node is `stale` iff any consumed port's current `port_manifest` version `>` the version recorded in `ranPortVersions`. Seeded from `notebook_dag_status`; kept fresh by `dag_status_changed`.

## Node State Machine

```mermaid
stateDiagram-v2
  [*] --> never_run
  never_run --> running: run
  running --> fresh: success
  running --> failed: error / raise
  fresh --> stale: consumed port version bumped upstream
  stale --> running: run / cascade
  fresh --> running: manual re-run
  failed --> running: re-run
  running --> upstream_failed: a producer failed first
  upstream_failed --> running: producer fixed, re-run
  failed --> [*]
```

- `stale` is *derived* (FE) from port versions; the others are *reported* (BE) via `CellRunReport` / `dag_status_changed`.
- `upstream_failed` mirrors the engine's existing `CellRunStatus::UpstreamFailed` (blocked because a producer in the cascade failed).

## Data Flow — the live cascade

```mermaid
sequenceDiagram
  actor U as User
  participant I as DagInspector
  participant S as Notebook store
  participant R as run_cell_and_cascade (Rust, NEW)
  participant E as ReactiveEngine
  participant Ev as notebook event channel
  U->>I: edit code, click "Run node"
  I->>S: optimistic editBuffer update (write_cell)
  I->>R: run_cell_and_cascade(cell_id)
  R->>E: run target cell; on success bump produced port manifest versions
  E->>E: stale_from_port(produced) -> ordered downstream
  loop each downstream (debounce 150ms, max_in_flight 4)
    E-->>Ev: dag_status_changed { id, running }
    E->>E: run cell (retry on stale_version x3)
    E-->>Ev: dag_status_changed { id, fresh | failed | upstream-failed }
  end
  Ev-->>S: status deltas applied to dagStatus slice
  S-->>I: React Flow nodes + edges recolor live
```

"Run node" (target only, no cascade) maps to the existing `run_cell` tool. "Run downstream" and the header "Run stale (N)" require the cascade entrypoint below.

## Backend Work (Rust)

The engine today only cascades from **datasource pushes** (`process_source_push` → `stale_from_source`). Two additions are needed:

### 1. `run_cell_and_cascade` entrypoint (the flagged gap)
A way to run a *cell* node and cascade its downstream, reusing the existing machinery:
- Run the target cell (`run_cell`), and on success bump its produced ports' manifest versions.
- Compute downstream via the existing `stale_from_port` for each produced port (already implemented in `graph.rs`).
- Reuse the existing cascade loop (debounce, `max_in_flight`, `UpstreamFailed` blocking, `STALE_RETRY_LIMIT`).
- Surface as an MCP tool (e.g. `notebook_run_cascade`) callable from the inspector; "Run stale (N)" calls it for each currently-stale root.

> Design note: prefer factoring the cascade loop out of `process_source_push` into a shared `cascade_from(seeds)` so both source-push and cell-run paths share one implementation.

### 2. `dag_status_changed` push events
- Emit a status delta on the **existing notebook event channel** (the one the FE already subscribes to for cell updates) whenever: a cell enters `running`, a `CellRunReport` is produced, or port manifest versions change.
- Payload: `{ notebook_version, nodes: [{ id, state, execution_count }], port_manifest: { port: version } }` (delta or full snapshot — full snapshot is simpler and the graph is small).
- FE adds one handler alongside its existing cell-update handler to merge into the `dagStatus` slice.

`notebook_dag_status` (already implemented) is reused unchanged as the **seed** when the toggle opens DAG mode.

## Frontend Components — files to create / modify

### New: `src/ui/dag/`
| File | Responsibility | Depends on |
|------|----------------|-----------|
| `DagView.tsx` | Top-level; React Flow `<ReactFlow>` canvas + `DagInspector`; renders when `viewMode==='dag'` | useDagGraph, DagNode, DagInspector, store |
| `useDagGraph.ts` | Pure: cells+`dagMetadata` → `{nodes, edges}` + dagre top-down layout | `@dagrejs/dagre`, bindings |
| `DagNode.tsx` | Custom RF node: label, code preview, port chips (in/out + version), state border | dagStatus |
| `DagInspector.tsx` | Selected node: CodeMirror (reuse `CellInput`), ports, Run node / Run downstream | store, run tools |
| `dagStatus.ts` | Seed (`notebook_dag_status`) + apply push deltas; stale derivation | bindings |
| `HiddenCellsChip.tsx` | Count + popover list of non-DAG cells | store |
| `layout.ts` | dagre wrapper (rankdir TB, node sizing) | dagre |

### Modify
| File | Change |
|------|--------|
| `stores/notebook.ts` | `dagMetadata` on `NotebookCellState`; load it in `loadNotebookRootDraft`; `viewMode` + `dagStatus` slices |
| `ui/notebook/NotebookView.tsx` | branch on `viewMode`: `NotebookCells` vs `DagView` |
| `ui/notebook/NotebookHeader.tsx` | segmented `Notebook | DAG` toggle; keyboard shortcut (e.g. ⌘⇧G) |
| event subscription (where cell updates are handled) | add `dag_status_changed` handler |
| `package.json` | add `@xyflow/react`, `@dagrejs/dagre` |

## Build Sequence (phases)

1. **Plumb + toggle (no graph lib).** `dagMetadata` into store; `viewMode` slice; header toggle; `DagView` placeholder that lists DAG cells. Verifies data is present.
2. **Graph render.** Add React Flow + dagre. `useDagGraph` derives nodes/edges; `DagNode` renders structure (the diamond) with auto-layout, pan/zoom/minimap. Static states.
3. **Seed status (read-only inspector).** Call `notebook_dag_status` on open → `dagStatus`; color nodes; `DagInspector` shows read-only code + port versions + derived `stale`.
4. **Edit + run target.** CodeMirror in inspector (reuse `CellInput`); "Run node" via existing `run_cell`; optimistic edit buffer.
5. **Cascade + live (Rust).** Add `run_cell_and_cascade` (refactor shared `cascade_from`) + `dag_status_changed` push; FE handler merges deltas; nodes/edges recolor live; "Run downstream" + "Run stale (N)".
6. **Hidden-cells chip + polish.** Chip + popover; edge stale animation; empty-state (no DAG cells); error toasts.

Each phase is independently demoable. Phases 1–4 are FE-only; phase 5 is the only one touching Rust.

## Testing

- **`useDagGraph` (unit):** given cell metadata, asserts derived edges match `NotebookDag::edges()` for the diamond / fan-out / independent-branches cases already covered in `graph.rs` tests — FE and BE must agree on shape.
- **Stale derivation (unit):** port version bump > `ranPortVersions` ⇒ `stale`; equal ⇒ `fresh`.
- **`dagStatus` reducer (unit):** applying a `dag_status_changed` delta produces the expected state map; `running` → `fresh`/`failed` transitions.
- **Rust `cascade_from` (integration):** extend `tests/reactive_dag.rs` to cover the cell-run cascade path (not just source push), incl. `UpstreamFailed` blocking.
- **Component (RTL):** toggle switches view; clicking a node selects it; "Run node" dispatches the right tool call.

## Open Questions / Risks
- **R1 — event channel shape.** Need to confirm the exact existing FE event subscription so `dag_status_changed` rides the same path. (Code-explore: `agent/handlers.ts`, `src-tauri/src/notebook_store.rs`.)
- **R2 — running-state granularity.** `run_cell` is synchronous over MCP; the engine must emit `running` *before* dispatch for the pulse to show. Confirm emit ordering.
- **R3 — `queued`/`debounced`.** Deferred from the state set; revisit if the 150ms debounce window feels invisible in practice.
- **R4 — React Flow bundle size.** Acceptable per decision #5; revisit if it regresses notebook cold-start.